In [18]:
import random
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
import numpy as np
import torch.utils.data
import cv2
import torchvision.models.segmentation
import torch
import os

In [19]:
batchSize=2
imageSize=[600,600]
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')   # train on the GPU or on the CPU, if a GPU is not available
trainDir="./Datasets/LabPics Chemistry/Train"

imgs=[]
for pth in os.listdir(trainDir):
    imgs .append(trainDir+"/"+pth +"//")
device

device(type='cpu')

In [20]:
imgs[:10]

['./Datasets/LabPics Chemistry/Train/959Train//',
 './Datasets/LabPics Chemistry/Train/3951Train//',
 './Datasets/LabPics Chemistry/Train/2643Train//',
 './Datasets/LabPics Chemistry/Train/2213Train//',
 './Datasets/LabPics Chemistry/Train/1595Train//',
 './Datasets/LabPics Chemistry/Train/4797Train//',
 './Datasets/LabPics Chemistry/Train/3052Train//',
 './Datasets/LabPics Chemistry/Train/3402Train//',
 './Datasets/LabPics Chemistry/Train/3117Train//',
 './Datasets/LabPics Chemistry/Train/4328Train//']

In [21]:
def loadData():
    batch_Imgs=[]
    batch_Data=[]# load images and masks
    for i in range(batchSize):
        idx=random.randint(0,len(imgs)-1)
        img = cv2.imread(os.path.join(imgs[idx], "Image.jpg"))
        img = cv2.resize(img, imageSize, cv2.INTER_LINEAR)
        maskDir=os.path.join(imgs[idx], "Vessels")
        masks=[]
        for mskName in os.listdir(maskDir):
            vesMask = (cv2.imread(maskDir+'/'+mskName, 0) > 0).astype(np.uint8)  # Read vesse instance mask
            vesMask=cv2.resize(vesMask,imageSize,cv2.INTER_NEAREST)
            masks.append(vesMask)# get bounding box coordinates for each mask
        num_objs = len(masks)
        if num_objs==0: return loadData() # if image have no objects just load another image
        boxes = torch.zeros([num_objs,4], dtype=torch.float32)
        for i in range(num_objs):
            x,y,w,h = cv2.boundingRect(masks[i])
            boxes[i] = torch.tensor([x, y, x+w, y+h])
        masks = torch.as_tensor(masks, dtype=torch.uint8)
        img = torch.as_tensor(img, dtype=torch.float32)
        data = {}
        data["boxes"] =  boxes
        data["labels"] =  torch.ones((num_objs,), dtype=torch.int64)   # there is only one class
        data["masks"] = masks
        batch_Imgs.append(img)
        batch_Data.append(data)  # load images and masks
    batch_Imgs = torch.stack([torch.as_tensor(d) for d in batch_Imgs], 0)
    batch_Imgs = batch_Imgs.swapaxes(1, 3).swapaxes(2, 3)
    return batch_Imgs, batch_Data

In [22]:
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)  # load an instance segmentation model pre-trained pre-trained on COCO
in_features = model.roi_heads.box_predictor.cls_score.in_features  # get number of input features for the classifier
model.roi_heads.box_predictor = FastRCNNPredictor(in_features,num_classes=2)  # replace the pre-trained head with a new one
model.to(device)# move model to the right devic

optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-5)
model.train()

MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [23]:
for i in range(10001):
            images, targets = loadData()
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            optimizer.zero_grad()
            loss_dict = model(images, targets)

            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            optimizer.step()
            print(i,'loss:', losses.item())
            if i%500==0:
                torch.save(model.state_dict(), str(i)+".torch")

0 loss: 137.01187133789062
1 loss: 189.76939392089844
2 loss: 44.03423309326172
3 loss: 107.86503601074219
4 loss: 17.360557556152344
5 loss: 19.899545669555664
6 loss: 22.703414916992188
7 loss: 24.6285400390625
8 loss: 9.677288055419922
9 loss: 23.369104385375977
10 loss: 27.605247497558594
11 loss: 47.8600959777832
12 loss: 10.938438415527344
13 loss: 21.603622436523438
14 loss: 13.96622371673584
15 loss: 16.741317749023438
16 loss: 13.690317153930664
17 loss: 11.943097114562988
18 loss: 30.252277374267578
19 loss: 20.659671783447266
20 loss: 11.882987022399902
21 loss: 9.470613479614258
22 loss: 19.716367721557617
23 loss: 15.002447128295898
24 loss: 11.86269474029541
25 loss: 8.515449523925781
26 loss: 8.89606761932373
27 loss: 10.10671329498291
28 loss: 21.904062271118164
29 loss: 8.95273208618164
30 loss: 5.9715070724487305
31 loss: 18.925683975219727
32 loss: 8.396185874938965
33 loss: 44.178585052490234
34 loss: 6.897339344024658
35 loss: 5.8786821365356445
36 loss: 9.00754833

KeyboardInterrupt: 